# Week 3.3: Keeping Your Twin Out of Trouble
### Plain-English edition — no coding required

> ▶ **Do this first: click _Connect_ at the top right, now.** Colab takes 10-30 seconds to
> start a computer for you. Let it happen while you read - by the time you reach Step 1
> it will be ready, and every step will run in about a second.

> **How do you stop an AI saying something it should not?**

You cannot. Not reliably, not by asking nicely in the instructions. What you *can* do is put
**gates** either side of it: check what goes in, and check what comes out, before anyone sees it.

Those gates are called **guardrails**. This notebook builds four of them and then shows you
their real cost — because a gate that blocks bad answers also blocks some good ones.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **Why instructions are not enough** — the twin leaks something it was never asked to
- **Topic filters** — refusing subjects you have decided are off limits
- **Personal data** — spotting and removing emails, phones and card numbers
- **The output validator** — every gate a reply must pass
- **Disclaimers** — medical, legal and financial answers need a warning attached
- **The full pipeline** — input gate, twin, output gate
- **The cost** — watching a perfectly good answer get blocked

---
## Part 1: The problem, in one step

Here is a twin with no guardrails at all. Its instructions say to be helpful. It is.

In [4]:
#@title ▶ Step 1 — Watch an unguarded twin leak something { display-mode: "form" }
message = "What time do you open?" #@param ["How can I contact you?", "How do I pay for my order?", "What should I take for a headache?", "What time do you open?"] {allow-input: true}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

PII_PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number":  r"(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){9,13}\d",
 "card number":   r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
 "id number":     r"\b\d{3}-\d{2}-\d{4}\b",
}
DISCLAIMERS = {
 "medical":   "Note: this is not medical advice. Please speak to a healthcare professional.",
 "legal":     "Note: this is not legal advice. Please speak to a lawyer.",
 "financial": "Note: this is not financial advice. Please speak to a financial adviser.",
}
PII_ORDER = ["card number", "id number", "email address", "phone number"]
def find_pii(text):
    """Scan in priority order, blanking what is already claimed so a card number
    is not also reported as a phone number."""
    working = str(text); found = {}
    for name in PII_ORDER:
        hits = []
        def take(m):
            s = m.group(0)
            if name == "phone number" and len(re.sub(r"\D", "", s)) < 9:
                return s
            hits.append(s.strip())
            return " " * len(s)
        working = re.sub(PII_PATTERNS[name], take, working)
        if hits: found[name] = hits
    return found
def redact(text):
    out = str(text)
    for name in PII_ORDER:
        tag = "[" + name.upper().replace(" ", "_") + " REMOVED]"
        if name == "phone number":
            out = re.sub(PII_PATTERNS[name],
                         lambda m: tag if len(re.sub(r"\D", "", m.group(0))) >= 9 else m.group(0), out)
        else:
            out = re.sub(PII_PATTERNS[name], tag, out)
    return out
def blocked_words():
    return globals().get("_BLOCKED", ["violence", "illegal", "weapon"])
def check_topics(text):
    low = str(text).lower()
    return [w for w in blocked_words() if w and w.lower() in low]
def needs_disclaimer(text):
    low = str(text).lower()
    return [k for k in DISCLAIMERS if k in low]
def validate_output(reply, max_length=400, check_pii=True, check_topics_on=True,
                    check_length=True, check_empty=True, add_disclaimers=True):
    """Every gate a reply must pass before a human sees it."""
    issues = []
    if check_empty and not str(reply).strip():
        issues.append("the reply is empty")
    if check_length and len(str(reply)) > max_length:
        issues.append("too long: " + str(len(str(reply))) + " characters, limit is " + str(max_length))
    if check_topics_on:
        bad = check_topics(reply)
        if bad: issues.append("mentions a blocked topic: " + ", ".join(bad))
    if check_pii:
        pii = find_pii(reply)
        if pii: issues.append("contains personal data: " + ", ".join(pii))
    topics = needs_disclaimer(reply) if add_disclaimers else []
    return {"ok": not issues, "issues": issues, "disclaimers": topics}
def fake_twin_reply(message):
    """Stands in for a real model, so the gates can be shown without an API key."""
    low = str(message).lower()
    if "contact" in low or "reach" in low:
        return "Sure - email me at sami.pharm@example.com or call +254 712 345 678."
    if "headache" in low or "medicine" in low or "dose" in low:
        return "For a headache most adults take paracetamol, but medical advice from a pharmacist is better."
    if "card" in low or "pay" in low:
        return "You can pay with the card 4111 1111 1111 1111 we have on file."
    if "weapon" in low or "illegal" in low:
        return "I can explain how illegal weapon sales usually work."
    if "essay" in low or "long" in low:
        return ("Here is everything I know about my pharmacy. " * 30).strip()
    return "My pharmacy opens at 7am and closes at 8pm, Monday to Saturday."

reply = fake_twin_reply(message)
print("SOMEONE ASKS : " + message)
print("")
box("THE TWIN REPLIES (no guardrails)", reply)
print("")
pii = find_pii(reply)
topics = needs_disclaimer(reply)
if pii:
    problem("That reply just published: " + ", ".join(pii))
    for name, hits in pii.items():
        for h in hits: print("   " + name + ": " + str(h))
    print("")
    print("Nobody asked it to keep that private, so it did not. It was being helpful.")
elif topics:
    problem("That reply gives " + topics[0] + " advice with no warning attached.")
    print("")
    print("Helpful, confident, and exactly the kind of answer that gets a product")
    print("into trouble.")
else:
    success("Nothing leaked this time.")
    print("")
    note("Try 'How can I contact you?' or 'How do I pay for my order?' and press play again.")
print("")
print("The fix is not a better instruction. It is a gate between the reply and the")
print("person reading it.")

SOMEONE ASKS : What time do you open?

+------------------------------------------------------------------------+
| THE TWIN REPLIES (no guardrails)                                       |
+------------------------------------------------------------------------+
| My pharmacy opens at 7am and closes at 8pm, Monday to Saturday.        |
+------------------------------------------------------------------------+




The fix is not a better instruction. It is a gate between the reply and the
person reading it.


---
## Part 2: Blocking topics you have ruled out

The simplest gate: a list of words that must not appear. Crude, fast, and the first thing
every real system does.

In [5]:
#@title ▶ Step 2 — Block the topics I choose { display-mode: "form" }
#@markdown Words that must not appear (separated by commas):
blocked_list = "violence, illegal, weapon" #@param {type:"string"}
#@markdown Something to check:
text_to_check = "I can explain how illegal weapon sales usually work." #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

PII_PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number":  r"(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){9,13}\d",
 "card number":   r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
 "id number":     r"\b\d{3}-\d{2}-\d{4}\b",
}
DISCLAIMERS = {
 "medical":   "Note: this is not medical advice. Please speak to a healthcare professional.",
 "legal":     "Note: this is not legal advice. Please speak to a lawyer.",
 "financial": "Note: this is not financial advice. Please speak to a financial adviser.",
}
PII_ORDER = ["card number", "id number", "email address", "phone number"]
def find_pii(text):
    """Scan in priority order, blanking what is already claimed so a card number
    is not also reported as a phone number."""
    working = str(text); found = {}
    for name in PII_ORDER:
        hits = []
        def take(m):
            s = m.group(0)
            if name == "phone number" and len(re.sub(r"\D", "", s)) < 9:
                return s
            hits.append(s.strip())
            return " " * len(s)
        working = re.sub(PII_PATTERNS[name], take, working)
        if hits: found[name] = hits
    return found
def redact(text):
    out = str(text)
    for name in PII_ORDER:
        tag = "[" + name.upper().replace(" ", "_") + " REMOVED]"
        if name == "phone number":
            out = re.sub(PII_PATTERNS[name],
                         lambda m: tag if len(re.sub(r"\D", "", m.group(0))) >= 9 else m.group(0), out)
        else:
            out = re.sub(PII_PATTERNS[name], tag, out)
    return out
def blocked_words():
    return globals().get("_BLOCKED", ["violence", "illegal", "weapon"])
def check_topics(text):
    low = str(text).lower()
    return [w for w in blocked_words() if w and w.lower() in low]
def needs_disclaimer(text):
    low = str(text).lower()
    return [k for k in DISCLAIMERS if k in low]
def validate_output(reply, max_length=400, check_pii=True, check_topics_on=True,
                    check_length=True, check_empty=True, add_disclaimers=True):
    """Every gate a reply must pass before a human sees it."""
    issues = []
    if check_empty and not str(reply).strip():
        issues.append("the reply is empty")
    if check_length and len(str(reply)) > max_length:
        issues.append("too long: " + str(len(str(reply))) + " characters, limit is " + str(max_length))
    if check_topics_on:
        bad = check_topics(reply)
        if bad: issues.append("mentions a blocked topic: " + ", ".join(bad))
    if check_pii:
        pii = find_pii(reply)
        if pii: issues.append("contains personal data: " + ", ".join(pii))
    topics = needs_disclaimer(reply) if add_disclaimers else []
    return {"ok": not issues, "issues": issues, "disclaimers": topics}
def fake_twin_reply(message):
    """Stands in for a real model, so the gates can be shown without an API key."""
    low = str(message).lower()
    if "contact" in low or "reach" in low:
        return "Sure - email me at sami.pharm@example.com or call +254 712 345 678."
    if "headache" in low or "medicine" in low or "dose" in low:
        return "For a headache most adults take paracetamol, but medical advice from a pharmacist is better."
    if "card" in low or "pay" in low:
        return "You can pay with the card 4111 1111 1111 1111 we have on file."
    if "weapon" in low or "illegal" in low:
        return "I can explain how illegal weapon sales usually work."
    if "essay" in low or "long" in low:
        return ("Here is everything I know about my pharmacy. " * 30).strip()
    return "My pharmacy opens at 7am and closes at 8pm, Monday to Saturday."

words = [w.strip() for w in blocked_list.split(",") if w.strip()]
globals()["_BLOCKED"] = words
if not words:
    problem("Your blocked list is empty, so nothing will ever be caught.")
else:
    print("Blocked words (" + str(len(words)) + "): " + ", ".join(words))
    print("")
    show_text("Checking:", text_to_check)
    print("")
    hits = check_topics(text_to_check)
    if hits:
        problem("BLOCKED - found: " + ", ".join(hits))
        print("")
        print("What the person would see instead:")
        box("", "I cannot help with that. Is there something else I can do?")
    else:
        success("ALLOWED - none of the blocked words appear.")
    print("")
    print("Be aware of what this does badly: it matches letters, not meaning.")
    note("'violence' also blocks 'non-violence' and 'domestic violence helpline'. "
         "Step 7 is about that cost.")

Blocked words (3): violence, illegal, weapon

Checking:
   I can explain how illegal weapon sales usually work.




What the person would see instead:
+------------------------------------------------------------------------+
|                                                                        |
+------------------------------------------------------------------------+
| I cannot help with that. Is there something else I can do?             |
+------------------------------------------------------------------------+

Be aware of what this does badly: it matches letters, not meaning.


---
## Part 3: Personal data

The gate that matters most in practice. Emails, phone numbers, card numbers and ID numbers
have recognisable **shapes**, so they can be found without understanding the text at all.

In [7]:
#@title ▶ Step 3 — Find and remove personal data { display-mode: "form" }
#@markdown Text to check (this is made-up data, not anyone's real details):
text_to_check = "Email me at sami.pharm@example.com or call +254 712 345 678. Card on file: 4111 1111 1111 1111." #@param {type:"string"}
#@markdown What should I do when I find some?
action = "block the whole reply" #@param ["remove it and keep the rest", "block the whole reply"]

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

PII_PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number":  r"(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){9,13}\d",
 "card number":   r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
 "id number":     r"\b\d{3}-\d{2}-\d{4}\b",
}
DISCLAIMERS = {
 "medical":   "Note: this is not medical advice. Please speak to a healthcare professional.",
 "legal":     "Note: this is not legal advice. Please speak to a lawyer.",
 "financial": "Note: this is not financial advice. Please speak to a financial adviser.",
}
PII_ORDER = ["card number", "id number", "email address", "phone number"]
def find_pii(text):
    """Scan in priority order, blanking what is already claimed so a card number
    is not also reported as a phone number."""
    working = str(text); found = {}
    for name in PII_ORDER:
        hits = []
        def take(m):
            s = m.group(0)
            if name == "phone number" and len(re.sub(r"\D", "", s)) < 9:
                return s
            hits.append(s.strip())
            return " " * len(s)
        working = re.sub(PII_PATTERNS[name], take, working)
        if hits: found[name] = hits
    return found
def redact(text):
    out = str(text)
    for name in PII_ORDER:
        tag = "[" + name.upper().replace(" ", "_") + " REMOVED]"
        if name == "phone number":
            out = re.sub(PII_PATTERNS[name],
                         lambda m: tag if len(re.sub(r"\D", "", m.group(0))) >= 9 else m.group(0), out)
        else:
            out = re.sub(PII_PATTERNS[name], tag, out)
    return out
def blocked_words():
    return globals().get("_BLOCKED", ["violence", "illegal", "weapon"])
def check_topics(text):
    low = str(text).lower()
    return [w for w in blocked_words() if w and w.lower() in low]
def needs_disclaimer(text):
    low = str(text).lower()
    return [k for k in DISCLAIMERS if k in low]
def validate_output(reply, max_length=400, check_pii=True, check_topics_on=True,
                    check_length=True, check_empty=True, add_disclaimers=True):
    """Every gate a reply must pass before a human sees it."""
    issues = []
    if check_empty and not str(reply).strip():
        issues.append("the reply is empty")
    if check_length and len(str(reply)) > max_length:
        issues.append("too long: " + str(len(str(reply))) + " characters, limit is " + str(max_length))
    if check_topics_on:
        bad = check_topics(reply)
        if bad: issues.append("mentions a blocked topic: " + ", ".join(bad))
    if check_pii:
        pii = find_pii(reply)
        if pii: issues.append("contains personal data: " + ", ".join(pii))
    topics = needs_disclaimer(reply) if add_disclaimers else []
    return {"ok": not issues, "issues": issues, "disclaimers": topics}
def fake_twin_reply(message):
    """Stands in for a real model, so the gates can be shown without an API key."""
    low = str(message).lower()
    if "contact" in low or "reach" in low:
        return "Sure - email me at sami.pharm@example.com or call +254 712 345 678."
    if "headache" in low or "medicine" in low or "dose" in low:
        return "For a headache most adults take paracetamol, but medical advice from a pharmacist is better."
    if "card" in low or "pay" in low:
        return "You can pay with the card 4111 1111 1111 1111 we have on file."
    if "weapon" in low or "illegal" in low:
        return "I can explain how illegal weapon sales usually work."
    if "essay" in low or "long" in low:
        return ("Here is everything I know about my pharmacy. " * 30).strip()
    return "My pharmacy opens at 7am and closes at 8pm, Monday to Saturday."

found = find_pii(text_to_check)
show_text("Before:", text_to_check)
print("")
if not found:
    success("No personal data found in that text.")
    note("Try the example above, which contains an email, a phone number and a card number.")
else:
    problem("Found " + str(sum(len(v) for v in found.values())) + " piece(s) of personal data:")
    for name, hits in found.items():
        for h in hits:
            print("   " + name.ljust(15) + " " + str(h))
    print("")
    if action == "block the whole reply":
        print("After (whole reply blocked):")
        box("", "I cannot share that information.")
        print("")
        note("Safest, but the person gets nothing - even the harmless part of the answer.")
    else:
        print("After:")
        show_text("", redact(text_to_check))
        print("")
        success("The personal data is gone and the rest of the message still works.")
    print("")
    print("Why shapes and not meaning: a card number is four groups of four digits")
    print("whatever language it sits in. No understanding required.")

Before:
   Email me at sami.pharm@example.com or call +254 712 345 678. Card on file:
   4111 1111 1111 1111.



   card number     4111 1111 1111 1111
   email address   sami.pharm@example.com
   phone number    +254 712 345 678

After (whole reply blocked):
+------------------------------------------------------------------------+
|                                                                        |
+------------------------------------------------------------------------+
| I cannot share that information.                                       |
+------------------------------------------------------------------------+




Why shapes and not meaning: a card number is four groups of four digits
whatever language it sits in. No understanding required.


---
## Part 4: The output validator

In a real system every reply passes through one checkpoint that runs all the gates at once
and either releases the reply or replaces it.

In [12]:
#@title ▶ Step 4 — Put a reply through every gate { display-mode: "form" }
reply_to_check = "Sure - email me at sami.pharm@example.com or call +254 712 345 678." #@param {type:"string"}
#@markdown Which gates should run?
reject_personal_data = True #@param {type:"boolean"}
reject_blocked_topics = True #@param {type:"boolean"}
reject_over_length = True #@param {type:"boolean"}
reject_empty = True #@param {type:"boolean"}
#@markdown Longest reply allowed (characters):
max_length = 400 #@param {type:"slider", min:50, max:1000, step:50}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

PII_PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number":  r"(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){9,13}\d",
 "card number":   r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
 "id number":     r"\b\d{3}-\d{2}-\d{4}\b",
}
DISCLAIMERS = {
 "medical":   "Note: this is not medical advice. Please speak to a healthcare professional.",
 "legal":     "Note: this is not legal advice. Please speak to a lawyer.",
 "financial": "Note: this is not financial advice. Please speak to a financial adviser.",
}
PII_ORDER = ["card number", "id number", "email address", "phone number"]
def find_pii(text):
    """Scan in priority order, blanking what is already claimed so a card number
    is not also reported as a phone number."""
    working = str(text); found = {}
    for name in PII_ORDER:
        hits = []
        def take(m):
            s = m.group(0)
            if name == "phone number" and len(re.sub(r"\D", "", s)) < 9:
                return s
            hits.append(s.strip())
            return " " * len(s)
        working = re.sub(PII_PATTERNS[name], take, working)
        if hits: found[name] = hits
    return found
def redact(text):
    out = str(text)
    for name in PII_ORDER:
        tag = "[" + name.upper().replace(" ", "_") + " REMOVED]"
        if name == "phone number":
            out = re.sub(PII_PATTERNS[name],
                         lambda m: tag if len(re.sub(r"\D", "", m.group(0))) >= 9 else m.group(0), out)
        else:
            out = re.sub(PII_PATTERNS[name], tag, out)
    return out
def blocked_words():
    return globals().get("_BLOCKED", ["violence", "illegal", "weapon"])
def check_topics(text):
    low = str(text).lower()
    return [w for w in blocked_words() if w and w.lower() in low]
def needs_disclaimer(text):
    low = str(text).lower()
    return [k for k in DISCLAIMERS if k in low]
def validate_output(reply, max_length=400, check_pii=True, check_topics_on=True,
                    check_length=True, check_empty=True, add_disclaimers=True):
    """Every gate a reply must pass before a human sees it."""
    issues = []
    if check_empty and not str(reply).strip():
        issues.append("the reply is empty")
    if check_length and len(str(reply)) > max_length:
        issues.append("too long: " + str(len(str(reply))) + " characters, limit is " + str(max_length))
    if check_topics_on:
        bad = check_topics(reply)
        if bad: issues.append("mentions a blocked topic: " + ", ".join(bad))
    if check_pii:
        pii = find_pii(reply)
        if pii: issues.append("contains personal data: " + ", ".join(pii))
    topics = needs_disclaimer(reply) if add_disclaimers else []
    return {"ok": not issues, "issues": issues, "disclaimers": topics}
def fake_twin_reply(message):
    """Stands in for a real model, so the gates can be shown without an API key."""
    low = str(message).lower()
    if "contact" in low or "reach" in low:
        return "Sure - email me at sami.pharm@example.com or call +254 712 345 678."
    if "headache" in low or "medicine" in low or "dose" in low:
        return "For a headache most adults take paracetamol, but medical advice from a pharmacist is better."
    if "card" in low or "pay" in low:
        return "You can pay with the card 4111 1111 1111 1111 we have on file."
    if "weapon" in low or "illegal" in low:
        return "I can explain how illegal weapon sales usually work."
    if "essay" in low or "long" in low:
        return ("Here is everything I know about my pharmacy. " * 30).strip()
    return "My pharmacy opens at 7am and closes at 8pm, Monday to Saturday."

result = validate_output(reply_to_check, max_length, reject_personal_data,
                         reject_blocked_topics, reject_over_length, reject_empty)
box("The reply being checked", reply_to_check[:300] + ("..." if len(reply_to_check) > 300 else ""))
print("")
gates = [("personal data", reject_personal_data), ("blocked topics", reject_blocked_topics),
         ("length", reject_over_length), ("empty", reject_empty)]
print("Gates running: " + ", ".join(g for g, on in gates if on))
off = [g for g, on in gates if not on]
if off: note("Switched off: " + ", ".join(off))
print("")
if result["ok"]:
    success("RELEASED - this reply passed every gate that is switched on.")
    if result["disclaimers"]:
        print("")
        print("...but it needs a disclaimer, which gets added automatically:")
        print("")
        box("", reply_to_check + chr(10) + chr(10) + DISCLAIMERS[result["disclaimers"][0]])
else:
    problem("HELD BACK - " + str(len(result["issues"])) + " problem(s):")
    for issue in result["issues"]:
        print("   - " + issue)
    print("")
    print("What the person sees instead:")
    box("", "I am not able to answer that. Can I help with something else?")
    print("")
    note("Untick a gate above and press play again to watch the same reply get through.")

+------------------------------------------------------------------------+
| The reply being checked                                                |
+------------------------------------------------------------------------+
| Sure - email me at sami.pharm@example.com or call +254 712 345 678.    |
+------------------------------------------------------------------------+

Gates running: personal data, blocked topics, length, empty



   - contains personal data: email address, phone number

What the person sees instead:
+------------------------------------------------------------------------+
|                                                                        |
+------------------------------------------------------------------------+
| I am not able to answer that. Can I help with something else?          |
+------------------------------------------------------------------------+



---
## Part 5: Disclaimers

Some answers are not unsafe, but they must not be mistaken for professional advice.
Your twin is a pharmacist's twin, so this one matters.

In [13]:
#@title ▶ Step 5 — Attach a warning where one is needed { display-mode: "form" }
question = "What should I take for a headache?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

PII_PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number":  r"(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){9,13}\d",
 "card number":   r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
 "id number":     r"\b\d{3}-\d{2}-\d{4}\b",
}
DISCLAIMERS = {
 "medical":   "Note: this is not medical advice. Please speak to a healthcare professional.",
 "legal":     "Note: this is not legal advice. Please speak to a lawyer.",
 "financial": "Note: this is not financial advice. Please speak to a financial adviser.",
}
PII_ORDER = ["card number", "id number", "email address", "phone number"]
def find_pii(text):
    """Scan in priority order, blanking what is already claimed so a card number
    is not also reported as a phone number."""
    working = str(text); found = {}
    for name in PII_ORDER:
        hits = []
        def take(m):
            s = m.group(0)
            if name == "phone number" and len(re.sub(r"\D", "", s)) < 9:
                return s
            hits.append(s.strip())
            return " " * len(s)
        working = re.sub(PII_PATTERNS[name], take, working)
        if hits: found[name] = hits
    return found
def redact(text):
    out = str(text)
    for name in PII_ORDER:
        tag = "[" + name.upper().replace(" ", "_") + " REMOVED]"
        if name == "phone number":
            out = re.sub(PII_PATTERNS[name],
                         lambda m: tag if len(re.sub(r"\D", "", m.group(0))) >= 9 else m.group(0), out)
        else:
            out = re.sub(PII_PATTERNS[name], tag, out)
    return out
def blocked_words():
    return globals().get("_BLOCKED", ["violence", "illegal", "weapon"])
def check_topics(text):
    low = str(text).lower()
    return [w for w in blocked_words() if w and w.lower() in low]
def needs_disclaimer(text):
    low = str(text).lower()
    return [k for k in DISCLAIMERS if k in low]
def validate_output(reply, max_length=400, check_pii=True, check_topics_on=True,
                    check_length=True, check_empty=True, add_disclaimers=True):
    """Every gate a reply must pass before a human sees it."""
    issues = []
    if check_empty and not str(reply).strip():
        issues.append("the reply is empty")
    if check_length and len(str(reply)) > max_length:
        issues.append("too long: " + str(len(str(reply))) + " characters, limit is " + str(max_length))
    if check_topics_on:
        bad = check_topics(reply)
        if bad: issues.append("mentions a blocked topic: " + ", ".join(bad))
    if check_pii:
        pii = find_pii(reply)
        if pii: issues.append("contains personal data: " + ", ".join(pii))
    topics = needs_disclaimer(reply) if add_disclaimers else []
    return {"ok": not issues, "issues": issues, "disclaimers": topics}
def fake_twin_reply(message):
    """Stands in for a real model, so the gates can be shown without an API key."""
    low = str(message).lower()
    if "contact" in low or "reach" in low:
        return "Sure - email me at sami.pharm@example.com or call +254 712 345 678."
    if "headache" in low or "medicine" in low or "dose" in low:
        return "For a headache most adults take paracetamol, but medical advice from a pharmacist is better."
    if "card" in low or "pay" in low:
        return "You can pay with the card 4111 1111 1111 1111 we have on file."
    if "weapon" in low or "illegal" in low:
        return "I can explain how illegal weapon sales usually work."
    if "essay" in low or "long" in low:
        return ("Here is everything I know about my pharmacy. " * 30).strip()
    return "My pharmacy opens at 7am and closes at 8pm, Monday to Saturday."

reply = fake_twin_reply(question)
print("SOMEONE ASKS : " + question)
print("")
topics = needs_disclaimer(reply)
if not topics:
    box("Reply (released as-is)", reply)
    print("")
    success("Nothing in this reply needs a warning.")
    note("Try 'What should I take for a headache?' to see one attached.")
else:
    box("Reply before", reply)
    print("")
    problem("This reply touches " + topics[0] + " advice.")
    print("")
    box("Reply after", reply + chr(10) + chr(10) + DISCLAIMERS[topics[0]])
    print("")
    success("The warning is added automatically - nobody has to remember.")
    print("")
    print("This is a gate that ADDS rather than blocks. The answer is still useful;")
    print("it just no longer reads like professional advice.")

SOMEONE ASKS : What should I take for a headache?

+------------------------------------------------------------------------+
| Reply before                                                           |
+------------------------------------------------------------------------+
| For a headache most adults take paracetamol, but medical advice from   |
| a pharmacist is better.                                                |
+------------------------------------------------------------------------+




+------------------------------------------------------------------------+
| Reply after                                                            |
+------------------------------------------------------------------------+
| For a headache most adults take paracetamol, but medical advice from   |
| a pharmacist is better.                                                |
|                                                                        |
| Note: this is not medical advice. Please speak to a healthcare         |
| professional.                                                          |
+------------------------------------------------------------------------+




This is a gate that ADDS rather than blocks. The answer is still useful;
it just no longer reads like professional advice.


---
## Part 6: The whole pipeline

Three stages: check what comes in, let the twin reply, check what goes out.
Most real failures happen because only one of those is in place.

In [19]:
#@title ▶ Step 6 — Run a message through the full pipeline { display-mode: "form" }
message = "How do I pay for my order?" #@param ["How do I pay for my order?", "How can I contact you?", "What should I take for a headache?", "Tell me about illegal weapon sales", "What time do you open?"] {allow-input: true}
gate_the_input = True #@param {type:"boolean"}
gate_the_output = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

PII_PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number":  r"(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){9,13}\d",
 "card number":   r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
 "id number":     r"\b\d{3}-\d{2}-\d{4}\b",
}
DISCLAIMERS = {
 "medical":   "Note: this is not medical advice. Please speak to a healthcare professional.",
 "legal":     "Note: this is not legal advice. Please speak to a lawyer.",
 "financial": "Note: this is not financial advice. Please speak to a financial adviser.",
}
PII_ORDER = ["card number", "id number", "email address", "phone number"]
def find_pii(text):
    """Scan in priority order, blanking what is already claimed so a card number
    is not also reported as a phone number."""
    working = str(text); found = {}
    for name in PII_ORDER:
        hits = []
        def take(m):
            s = m.group(0)
            if name == "phone number" and len(re.sub(r"\D", "", s)) < 9:
                return s
            hits.append(s.strip())
            return " " * len(s)
        working = re.sub(PII_PATTERNS[name], take, working)
        if hits: found[name] = hits
    return found
def redact(text):
    out = str(text)
    for name in PII_ORDER:
        tag = "[" + name.upper().replace(" ", "_") + " REMOVED]"
        if name == "phone number":
            out = re.sub(PII_PATTERNS[name],
                         lambda m: tag if len(re.sub(r"\D", "", m.group(0))) >= 9 else m.group(0), out)
        else:
            out = re.sub(PII_PATTERNS[name], tag, out)
    return out
def blocked_words():
    return globals().get("_BLOCKED", ["violence", "illegal", "weapon"])
def check_topics(text):
    low = str(text).lower()
    return [w for w in blocked_words() if w and w.lower() in low]
def needs_disclaimer(text):
    low = str(text).lower()
    return [k for k in DISCLAIMERS if k in low]
def validate_output(reply, max_length=400, check_pii=True, check_topics_on=True,
                    check_length=True, check_empty=True, add_disclaimers=True):
    """Every gate a reply must pass before a human sees it."""
    issues = []
    if check_empty and not str(reply).strip():
        issues.append("the reply is empty")
    if check_length and len(str(reply)) > max_length:
        issues.append("too long: " + str(len(str(reply))) + " characters, limit is " + str(max_length))
    if check_topics_on:
        bad = check_topics(reply)
        if bad: issues.append("mentions a blocked topic: " + ", ".join(bad))
    if check_pii:
        pii = find_pii(reply)
        if pii: issues.append("contains personal data: " + ", ".join(pii))
    topics = needs_disclaimer(reply) if add_disclaimers else []
    return {"ok": not issues, "issues": issues, "disclaimers": topics}
def fake_twin_reply(message):
    """Stands in for a real model, so the gates can be shown without an API key."""
    low = str(message).lower()
    if "contact" in low or "reach" in low:
        return "Sure - email me at sami.pharm@example.com or call +254 712 345 678."
    if "headache" in low or "medicine" in low or "dose" in low:
        return "For a headache most adults take paracetamol, but medical advice from a pharmacist is better."
    if "card" in low or "pay" in low:
        return "You can pay with the card 4111 1111 1111 1111 we have on file."
    if "weapon" in low or "illegal" in low:
        return "I can explain how illegal weapon sales usually work."
    if "essay" in low or "long" in low:
        return ("Here is everything I know about my pharmacy. " * 30).strip()
    return "My pharmacy opens at 7am and closes at 8pm, Monday to Saturday."

print("MESSAGE IN   : " + message)
print("")
stopped = False
if gate_the_input:
    bad = check_topics(message)
    if bad:
        problem("GATE 1 (input)  : BLOCKED - mentions " + ", ".join(bad))
        stopped = True
    else:
        print("GATE 1 (input)  : passed")
else:
    note("GATE 1 (input)  : switched off")
if not stopped:
    reply = fake_twin_reply(message)
    print("TWIN REPLIES    : " + reply[:60] + ("..." if len(reply) > 60 else ""))
    if gate_the_output:
        result = validate_output(reply)
        if result["ok"]:
            print("GATE 2 (output) : passed")
            final = reply
            if result["disclaimers"]:
                final = reply + chr(10) + chr(10) + DISCLAIMERS[result["disclaimers"][0]]
                print("                  disclaimer added (" + result["disclaimers"][0] + ")")
        else:
            print("GATE 2 (output) : BLOCKED - " + "; ".join(result["issues"]))
            final = "I am not able to answer that. Can I help with something else?"
    else:
        note("GATE 2 (output) : switched off")
        final = reply
    print("")
    box("WHAT THE PERSON SEES", final)
    print("")
    if gate_the_output and not validate_output(reply)["ok"]:
        success("The gate caught it. The raw reply never reached anyone.")
    elif not gate_the_output and not validate_output(reply)["ok"]:
        problem("With the output gate off, that reply went straight out.")
        note("Switch 'gate the output' back on and press play again.")
else:
    print("")
    box("WHAT THE PERSON SEES", "I cannot help with that. Is there something else I can do?")
    print("")
    success("Blocked before the twin was even called - which also saves you the cost.")

MESSAGE IN   : How do I pay for my order?

GATE 1 (input)  : passed
TWIN REPLIES    : You can pay with the card 4111 1111 1111 1111 we have on fil...
GATE 2 (output) : BLOCKED - contains personal data: card number

+------------------------------------------------------------------------+
| WHAT THE PERSON SEES                                                   |
+------------------------------------------------------------------------+
| I am not able to answer that. Can I help with something else?          |
+------------------------------------------------------------------------+



---
## Part 7: What guardrails cost

Every gate makes two kinds of mistake. It lets something bad through, or it blocks something
perfectly good. You cannot have zero of both, so you choose which you would rather have.

In [24]:
#@title ▶ Step 7 — Watch a good answer get blocked { display-mode: "form" }
#@markdown A genuinely useful reply that trips a crude filter:
good_reply = "If you are affected by domestic violence, the free helpline is open 24 hours." #@param ["If you are affected by domestic violence, the free helpline is open 24 hours.", "Our pharmacy is at 12 Moi Avenue - call 041 2222 333 for directions.", "Non-violence training is held at the community centre on Tuesdays."] {allow-input: true}
blocked_list = "violence, illegal, weapon" #@param {type:"string"}
check_personal_data = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

PII_PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number":  r"(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){9,13}\d",
 "card number":   r"\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b",
 "id number":     r"\b\d{3}-\d{2}-\d{4}\b",
}
DISCLAIMERS = {
 "medical":   "Note: this is not medical advice. Please speak to a healthcare professional.",
 "legal":     "Note: this is not legal advice. Please speak to a lawyer.",
 "financial": "Note: this is not financial advice. Please speak to a financial adviser.",
}
PII_ORDER = ["card number", "id number", "email address", "phone number"]
def find_pii(text):
    """Scan in priority order, blanking what is already claimed so a card number
    is not also reported as a phone number."""
    working = str(text); found = {}
    for name in PII_ORDER:
        hits = []
        def take(m):
            s = m.group(0)
            if name == "phone number" and len(re.sub(r"\D", "", s)) < 9:
                return s
            hits.append(s.strip())
            return " " * len(s)
        working = re.sub(PII_PATTERNS[name], take, working)
        if hits: found[name] = hits
    return found
def redact(text):
    out = str(text)
    for name in PII_ORDER:
        tag = "[" + name.upper().replace(" ", "_") + " REMOVED]"
        if name == "phone number":
            out = re.sub(PII_PATTERNS[name],
                         lambda m: tag if len(re.sub(r"\D", "", m.group(0))) >= 9 else m.group(0), out)
        else:
            out = re.sub(PII_PATTERNS[name], tag, out)
    return out
def blocked_words():
    return globals().get("_BLOCKED", ["violence", "illegal", "weapon"])
def check_topics(text):
    low = str(text).lower()
    return [w for w in blocked_words() if w and w.lower() in low]
def needs_disclaimer(text):
    low = str(text).lower()
    return [k for k in DISCLAIMERS if k in low]
def validate_output(reply, max_length=400, check_pii=True, check_topics_on=True,
                    check_length=True, check_empty=True, add_disclaimers=True):
    """Every gate a reply must pass before a human sees it."""
    issues = []
    if check_empty and not str(reply).strip():
        issues.append("the reply is empty")
    if check_length and len(str(reply)) > max_length:
        issues.append("too long: " + str(len(str(reply))) + " characters, limit is " + str(max_length))
    if check_topics_on:
        bad = check_topics(reply)
        if bad: issues.append("mentions a blocked topic: " + ", ".join(bad))
    if check_pii:
        pii = find_pii(reply)
        if pii: issues.append("contains personal data: " + ", ".join(pii))
    topics = needs_disclaimer(reply) if add_disclaimers else []
    return {"ok": not issues, "issues": issues, "disclaimers": topics}
def fake_twin_reply(message):
    """Stands in for a real model, so the gates can be shown without an API key."""
    low = str(message).lower()
    if "contact" in low or "reach" in low:
        return "Sure - email me at sami.pharm@example.com or call +254 712 345 678."
    if "headache" in low or "medicine" in low or "dose" in low:
        return "For a headache most adults take paracetamol, but medical advice from a pharmacist is better."
    if "card" in low or "pay" in low:
        return "You can pay with the card 4111 1111 1111 1111 we have on file."
    if "weapon" in low or "illegal" in low:
        return "I can explain how illegal weapon sales usually work."
    if "essay" in low or "long" in low:
        return ("Here is everything I know about my pharmacy. " * 30).strip()
    return "My pharmacy opens at 7am and closes at 8pm, Monday to Saturday."

globals()["_BLOCKED"] = [w.strip() for w in blocked_list.split(",") if w.strip()]
box("A reply you would be happy to send", good_reply)
print("")
result = validate_output(good_reply, 400, check_personal_data, True, True, True)
if result["ok"]:
    success("Released - the gates let this one through, correctly.")
    note("Try one of the other examples in the dropdown to see a false alarm.")
else:
    problem("BLOCKED - " + "; ".join(result["issues"]))
    print("")
    print("Nothing is wrong with that reply. The filter matched letters, not meaning.")
    print("")
    print("This is called a FALSE POSITIVE, and it is the real cost of guardrails:")
    print("   - block too much, and your twin becomes useless and annoying")
    print("   - block too little, and it says something you have to apologise for")
    print("")
    print("There is no setting that avoids both. What you can do is decide which")
    print("mistake is cheaper for YOUR twin, and tune towards it deliberately.")
    print("")
    note("For a pharmacist's twin, blocking a helpline number is a serious failure. "
         "You would narrow 'violence' to a phrase, or check meaning rather than letters.")

+------------------------------------------------------------------------+
| A reply you would be happy to send                                     |
+------------------------------------------------------------------------+
| If you are affected by domestic violence, the free helpline is open    |
| 24 hours.                                                              |
+------------------------------------------------------------------------+




Nothing is wrong with that reply. The filter matched letters, not meaning.

This is called a FALSE POSITIVE, and it is the real cost of guardrails:
   - block too much, and your twin becomes useless and annoying
   - block too little, and it says something you have to apologise for

There is no setting that avoids both. What you can do is decide which
mistake is cheaper for YOUR twin, and tune towards it deliberately.



---
## What you just did

| Step | In plain words | The real name for it |
|---|---|---|
| 1 | Watched a helpful twin leak a phone number | Unguarded output |
| 2 | Refused certain topics | Content filtering |
| 3 | Found and removed personal details | PII detection and redaction |
| 4 | Ran every check in one place | Output validation |
| 5 | Attached a medical warning automatically | Disclaimers |
| 6 | Gated both the input and the output | The safety pipeline |
| 7 | Blocked a reply that was perfectly fine | False positives |

### The one sentence to remember
**Guardrails are code, not instructions.** Asking a model to behave is a request. A gate that
inspects every reply is a guarantee.

### Gate both ends
Input gates are cheaper — they stop a bad request before you pay for a reply. Output gates are
the ones that save you, because the twin can produce something you never anticipated from a
request that looked completely innocent.

### Where this is going
These gates assume people are asking in good faith. **Week 3.4** is about people who are not:
attacking your own twin on purpose, to find what gets through.

### Want to see the code?
Click **Show code** on any step.